# Model training

This stage involves training the ML algorithm by providing it with datasets, where the learning process takes place. Consistent training can significantly enhance the model's prediction accuracy. It's essential to initialize the model's weights randomly so the algorithm can effectively learn to adjust them.

# XGBoost

In [ ]:
from xgboost import XGBRFClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import randint

model = XGBRFClassifier(random_state=42)
params = {
    "learning_rate": uniform(1e-2, 3e-1),
    "min_split_loss": uniform(0, 10),
    "max_depth": randint(3, 10),
    "subsample": uniform(0, 1),
    "objective": ["reg:squarederror", "binary:logistic", "reg:logistic"],
    "eval_metric": ["aucpr", "error"]
}

model_grid = RandomizedSearchCV(model, param_distributions=params, n_jobs=-1, verbose=3, n_iter=10, cv=10)

model_grid.fit(X_train, y_train)

# Model test accuracy

In [ ]:
from sklearn.metrics import accuracy_score

best_model_xgboost_params = model_grid.best_params_
print("Best xgboost params")
pprint(best_model_xgboost_params)

y_pred_train = model_grid.predict(X_train)
y_pred_test = model_grid.predict(X_test)
print("Accuracy train", accuracy_score(y_pred_train, y_train ))
print("Accuracy test", accuracy_score(y_pred_test, y_test))


# XGBoost performance overview
* Confusion matrix
* Classification report

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

conf_matrix = confusion_matrix(y_test, y_pred_test)
print("Test actual/predicted\n")
print(pd.crosstab(y_test, y_pred_test, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_test, y_pred_test),'\n')

conf_matrix = confusion_matrix(y_train, y_pred_train)
print("Train actual/predicted\n")
print(pd.crosstab(y_train, y_pred_train, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_train, y_pred_train),'\n')

# Save best XGBoost model

In [ ]:
xgboost_model = model_grid.best_estimator_
xgboost_model_path = "./artifacts/lead_model_xgboost.json"
xgboost_model.save_model(xgboost_model_path)

model_results = {
    xgboost_model_path: classification_report(y_train, y_pred_train, output_dict=True)
}

# SKLearn logistic regression

In [ ]:
import mlflow.pyfunc

from sklearn.linear_model import LogisticRegression
import os
from sklearn.metrics import cohen_kappa_score, f1_score
import matplotlib.pyplot as plt
import joblib

class lr_wrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]


mlflow.sklearn.autolog(log_input_examples=True, log_models=False)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

with mlflow.start_run(experiment_id=experiment_id) as run:
    model = LogisticRegression()
    lr_model_path = "./artifacts/lead_model_lr.pkl"

    params = {
              'solver': ["newton-cg", "lbfgs", "liblinear", "sag", "saga"],
              'penalty':  ["none", "l1", "l2", "elasticnet"],
              'C' : [100, 10, 1.0, 0.1, 0.01]
    }
    model_grid = RandomizedSearchCV(model, param_distributions= params, verbose=3, n_iter=10, cv=3)
    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)


    # log artifacts
    mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    mlflow.log_artifacts("artifacts", artifact_path="model")
    mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    joblib.dump(value=model, filename=lr_model_path)
        
    # Custom python model for predicting probability 
    mlflow.pyfunc.log_model('model', python_model=lr_wrapper(model))


model_classification_report = classification_report(y_test, y_pred_test, output_dict=True)

best_model_lr_params = model_grid.best_params_

print("Best lr params")
pprint(best_model_lr_params)

print("Accuracy train:", accuracy_score(y_pred_train, y_train ))
print("Accuracy test:", accuracy_score(y_pred_test, y_test))

conf_matrix = confusion_matrix(y_test, y_pred_test)
print("Test actual/predicted\n")
print(pd.crosstab(y_test, y_pred_test, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_test, y_pred_test),'\n')

conf_matrix = confusion_matrix(y_train, y_pred_train)
print("Train actual/predicted\n")
print(pd.crosstab(y_train, y_pred_train, rownames=['Actual'], colnames=['Predicted'], margins=True),'\n')
print("Classification report\n")
print(classification_report(y_train, y_pred_train),'\n')

model_results[lr_model_path] = model_classification_report
print(model_classification_report["weighted avg"]["f1-score"])


# Save columns and model results

In [ ]:
column_list_path = './artifacts/columns_list.json'
with open(column_list_path, 'w+') as columns_file:
    columns = {'column_names': list(X_train.columns)}
    pprint(columns)
    json.dump(columns, columns_file)

print('Saved column list to ', column_list_path)

model_results_path = "./artifacts/model_results.json"
with open(model_results_path, 'w+') as results_file:
    json.dump(model_results, results_file)

# New code

In [1]:
max_date = "2024-01-31"
min_date = "2024-01-01"

In [2]:
import os
import shutil
from pprint import pprint

# shutil.rmtree("./artifacts",ignore_errors=True)
os.makedirs("artifacts",exist_ok=True)
print("Created artifacts directory")

Created artifacts directory


In [3]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format',lambda x: "%.3f" % x)

In [15]:
from xgboost import XGBRFClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import randint
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import mlflow.pyfunc
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score
import matplotlib.pyplot as plt
import joblib
import datetime

In [ ]:
dataX = pd.read_csv("./artifacts/X_test.csv")
datay = pd.read_csv("./artifacts/y_test.csv")

display(dataX.head(5))
display(datay.head(5))

,purchases,time_spent,n_visits,customer_group_2,customer_group_3,customer_group_4,customer_group_5,customer_group_6,customer_group_7,customer_group_8,customer_group_9,onboarding_True
0,0.730,0.153,0.239,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
1,0.294,0.482,0.479,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.294,0.469,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000
3,0.621,0.392,0.419,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.076,0.837,0.120,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000


,lead_indicator
0,0.000
1,1.000
2,0.000
3,1.000
4,0.000


12

In [12]:
#from sklearn.model_selection import train_test_split

#X_train, X_test, y_train, y_test = train_test_split(
#    X, y, random_state=42, test_size=0.15, stratify=y
#)
#y_train

X_train=dataX
y_train=datay
X_test=dataX
y_test=datay

In [ ]:
current_date = datetime.datetime.now().strftime("%Y_%B_%d")
artifact_path = "model"
model_name = "lead_model"
experiment_name = current_date

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
experiment_ids = [mlflow.get_experiment_by_name(experiment_name).experiment_id]
experiment_ids

AttributeError: 'NoneType' object has no attribute 'experiment_id'

In [13]:
class XGBWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]

/home/dimiko/MLOps/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [ ]:
model = XGBRFClassifier(random_state=42)
params = {
    "learning_rate": uniform(1e-2, 3e-1),
    "min_split_loss": uniform(0, 10),
    "max_depth": randint(3, 10),
    "subsample": uniform(0, 1),
    "objective": ["reg:squarederror", "binary:logistic", "reg:logistic"],
    "eval_metric": ["aucpr", "error"]
}

model_grid = RandomizedSearchCV(model, param_distributions=params, n_jobs=-1, verbose=3, n_iter=10, cv=10)

model_grid.fit(X_train, y_train)

In [19]:
with mlflow.start_run(experiment_id=None) as run:
    model = XGBRFClassifier(random_state=42)
    params = {
        "learning_rate": uniform(1e-2, 3e-1),
        "min_split_loss": uniform(0, 10),
        "max_depth": randint(3, 10),
        "subsample": uniform(0, 1),
        "objective": ["reg:squarederror", "binary:logistic", "reg:logistic"],
        "eval_metric": ["aucpr", "error"]
    }
    model_grid = RandomizedSearchCV(model, param_distributions=params, n_jobs=-1, verbose=3, n_iter=10, cv=10)

    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)

    print("Best xgboost params")
    pprint(model_grid.best_params_)
    print("Accuracy train", accuracy_score(y_pred_train, y_train ))
    print("Accuracy test", accuracy_score(y_pred_test, y_test))
    

    # log artifacts
    #mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    #mlflow.log_artifacts("artifacts", artifact_path="model")
    #mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    #joblib.dump(value=model, filename=lr_model_path)
        
    # Custom python model for predicting probability 
    #mlflow.pyfunc.log_model('model', python_model=lr_wrapper(model))

Fitting 10 folds for each of 10 candidates, totalling 100 fits
[CV 8/10] END eval_metric=error, learning_rate=0.18903954782143562, max_depth=7, min_split_loss=3.452024527151556, objective=reg:logistic, subsample=0.8458692394338757;, score=0.333 total time=   1.9s
[CV 6/10] END eval_metric=error, learning_rate=0.18903954782143562, max_depth=7, min_split_loss=3.452024527151556, objective=reg:logistic, subsample=0.8458692394338757;, score=0.500 total time=   1.9s
[CV 3/10] END eval_metric=error, learning_rate=0.22618700645501955, max_depth=4, min_split_loss=4.028400596386169, objective=binary:logistic, subsample=0.2801434048137098;, score=1.000 total time=   0.0s
[CV 4/10] END eval_metric=error, learning_rate=0.22618700645501955, max_depth=4, min_split_loss=4.028400596386169, objective=binary:logistic, subsample=0.2801434048137098;, score=0.750 total time=   0.0s
[CV 5/10] END eval_metric=error, learning_rate=0.22618700645501955, max_depth=4, min_split_loss=4.028400596386169, objective=bi

2025/11/01 23:49:48 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.


Best xgboost params
{'eval_metric': 'error',
 'learning_rate': np.float64(0.2617976000408778),
 'max_depth': 9,
 'min_split_loss': np.float64(2.628310021580522),
 'objective': 'reg:logistic',
 'subsample': np.float64(0.7858044285647809)}
Accuracy train 0.8055555555555556
Accuracy test 0.8055555555555556


In [20]:
class lr_wrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    
    def predict(self, context, model_input):
        return self.model.predict_proba(model_input)[:, 1]


/home/dimiko/MLOps/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [22]:
with mlflow.start_run(experiment_id=None) as run:
    model = LogisticRegression()
    lr_model_path = "./artifacts/lead_model_lr.pkl"

    params = {
              'solver': ["newton-cg", "lbfgs", "liblinear", "sag", "saga"],
              'penalty':  ["none", "l1", "l2", "elasticnet"],
              'C' : [100, 10, 1.0, 0.1, 0.01]
    }
    model_grid = RandomizedSearchCV(model, param_distributions= params, verbose=3, n_iter=10, cv=3)
    model_grid.fit(X_train, y_train)

    best_model = model_grid.best_estimator_

    y_pred_train = model_grid.predict(X_train)
    y_pred_test = model_grid.predict(X_test)

    print("Best lr params")
    pprint(model_grid.best_params_)
    print("Accuracy train", accuracy_score(y_pred_train, y_train ))
    print("Accuracy test", accuracy_score(y_pred_test, y_test))
    # log artifacts
    #mlflow.log_metric('f1_score', f1_score(y_test, y_pred_test))
    #mlflow.log_artifacts("artifacts", artifact_path="model")
    #mlflow.log_param("data_version", "00000")
    
    # store model for model interpretability
    #joblib.dump(value=model, filename=lr_model_path)
        
    # Custom python model for predicting probability 
    #mlflow.pyfunc.log_model('model', python_model=lr_wrapper(model))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV 1/3] END .......C=0.1, penalty=l1, solver=sag;, score=nan total time=   0.0s
[CV 2/3] END .......C=0.1, penalty=l1, solver=sag;, score=nan total time=   0.0s
[CV 3/3] END .......C=0.1, penalty=l1, solver=sag;, score=nan total time=   0.0s
[CV 1/3] END C=10, penalty=elasticnet, solver=liblinear;, score=nan total time=   0.0s
[CV 2/3] END C=10, penalty=elasticnet, solver=liblinear;, score=nan total time=   0.0s
[CV 3/3] END C=10, penalty=elasticnet, solver=liblinear;, score=nan total time=   0.0s
[CV 1/3] END .C=0.1, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 2/3] END .C=0.1, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 3/3] END .C=0.1, penalty=l1, solver=newton-cg;, score=nan total time=   0.0s
[CV 1/3] END .....C=1.0, penalty=none, solver=sag;, score=nan total time=   0.0s
[CV 2/3] END .....C=1.0, penalty=none, solver=sag;, score=nan total time=   0.0s
[CV 3/3] END .....C=1.0, penal

2025/11/01 23:51:15 INFO mlflow.sklearn.utils: Logging the 5 best runs, 5 runs will be omitted.


Best lr params
{'C': 0.1, 'penalty': 'l2', 'solver': 'newton-cg'}
Accuracy train 0.8333333333333334
Accuracy test 0.8333333333333334
